In [ ]:
from datetime import datetime
from glob import glob
import os
import numpy as np
import matplotlib.pyplot as plt
from eigsep_corr import io
%matplotlib inline

In [ ]:
DATA_DIR = "/home/christian/Documents/research/eigsep/data-analysis/data/night1"
# subset of files with us definitely gone
files = np.array(sorted(glob(os.path.join(DATA_DIR, "2023101[4-5]*")), key=os.path.getctime)[265:-45])

In [ ]:
def fname2ts(f):
    t = f[-len("20231014_202203.eig"):-4]
    y = t[:4]
    mo = t[4:6]
    d = t[6:8]
    h = t[9:11]
    mi = t[11:13]
    s = t[13:15]
    dt = datetime.fromisoformat(f"{y}-{mo}-{d} {h}:{mi}:{s}")
    return dt.timestamp()

ftime = np.empty(len(files))
for i, f in enumerate(files):
    ftime[i] = fname2ts(f)
ix = np.argsort(ftime)[200:]
ftime = ftime[ix]
files = files[ix]

In [ ]:
h = io.read_file(files[0])[0]
N_ACC = len(h["acc_cnt"])
ACC_BINS = h["acc_bins"]
NCHAN = h["nchan"]

acc_cnt = []
times = []
sync_time = []
t0s = []
for f in files:
    hdr = io.read_header(f)
    acc_cnt.append(hdr['acc_cnt'])
    times.append(hdr['times'])
    sync_time.append(hdr["sync_time"])
    t0s.append(hdr["times"][0])
acc_cnt = np.concatenate(acc_cnt)
times = np.concatenate(times)
t0s = np.array(t0s)
assert np.allclose(sync_time, sync_time[0])

print(np.allclose(np.diff(acc_cnt), 1))

In [ ]:
print((times[0] - ftime[0]) / 3600)

In [ ]:
plt.figure()
plt.plot(ftime-ftime[0])
plt.show()

In [ ]:
plt.figure()
plt.plot(acc_cnt, times-times[0])
plt.plot(acc_cnt[::60], ftime-ftime[0])
plt.show()

In [ ]:
plt.figure()
plt.plot(acc_cnt[::60], t0s-t0s[0])
plt.plot(acc_cnt[::60], ftime-ftime[0])
plt.show()

In [ ]:
t0s = t0s - t0s[0]
ftime = ftime - ftime[0]
t0s = t0s[1:]
ftime = ftime[1:]

In [ ]:
plt.figure()
plt.plot(ftime/t0s)
plt.xlim(200, 500)
plt.show()

In [ ]:
t0s = t0s[200:]
ftime = ftime[200:]
plt.figure()
plt.plot(t0s*2)
plt.plot(ftime, ls="--")
plt.show()

In [ ]:
hdr